# ⚡ Transvid: Neural Latent Media Compression Machine

Welcome to the official **Transvid** demonstration notebook!

Transvid encodes audio and video into **compact neural latent representations** (`.tvid` format) and reconstructs high-fidelity media back using deep neural decoders.

---
### 🚀 Quickstart: Run Cells in Order
1. **Step 1:** Verify T4 GPU is connected.
2. **Step 2:** Install dependencies.
3. **Step 3:** Clone or update the Transvid codebase.
4. **Step 4 & 5:** Compress & play Audio and Video!

### Step 1: Verify Cloud GPU Environment

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not detected! Go to Runtime -> Change runtime type -> Select T4 GPU")

### Step 2: Install Dependencies

In [ ]:
!pip install -q encodec torchaudio soundfile opencv-python-headless numpy zstandard tqdm

### Step 3: Clone or Pull Latest Transvid Code

In [ ]:
import os
if not os.path.exists("Transvid"):
    !git clone https://github.com/boklos2209-lang/Transvid.git
%cd Transvid
!git pull

## 🎵 Part 1: Audio Latent Compression (EnCodec RVQ)
Compress audio files (`.mp3`, `.wav`, etc.) into `.tvid` and reconstruct them back.

In [ ]:
import os
from IPython.display import Audio, display

# Enter your audio filename here (or use the uploaded Bruno.mp3)
audio_input = "../Bruno.mp3" if os.path.exists("../Bruno.mp3") else "demo_audio.wav"

# If no file uploaded, generate a 5-second musical sample
if not os.path.exists(audio_input):
    import numpy as np, torchaudio, torch
    sr, dur = 44100, 5.0
    t = np.linspace(0, dur, int(sr * dur))
    chord = (0.4*np.sin(2*np.pi*261.63*t) + 0.3*np.sin(2*np.pi*329.63*t) + 0.3*np.sin(2*np.pi*392.00*t)).astype(np.float32)
    torchaudio.save("demo_audio.wav", torch.from_numpy(chord).unsqueeze(0), sr)
    audio_input = "demo_audio.wav"

# 1. Compress to .tvid
!python -m src.cli compress "{audio_input}" -o "audio.tvid" --bandwidth 6.0

# 2. Decompress back to MP3
!python -m src.cli decompress "audio.tvid" -o "audio_restored.mp3"

# 3. Listen right here in Colab!
print("\n🎧 Reconstructed Audio Playback:")
display(Audio("audio_restored.mp3"))

## 🎬 Part 2: High-Efficiency Video Latent Compression
Compress video files (`.mp4`) into `.tvid` using compact 16x latents and reconstruct with synchronized audio.

In [ ]:
import os
from IPython.display import HTML
from base64 import b64encode

# Use your uploaded video or a demo
video_input = "../demo.mp4" if os.path.exists("../demo.mp4") else "demo_video.mp4"

# If no video uploaded, generate a test moving video
if not os.path.exists(video_input):
    import cv2, numpy as np
    w, h, fps, total = 128, 128, 30.0, 60
    vw = cv2.VideoWriter("demo_video.mp4", cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    for i in range(total):
        f = np.zeros((h, w, 3), dtype=np.uint8)
        f[:, :] = [i * 4 % 255, 120, 160]
        cv2.circle(f, (int(w/2 + 35*np.cos(i*0.15)), int(h/2 + 35*np.sin(i*0.15))), 18, (255, 255, 255), -1)
        cv2.putText(f, f"Frame {i}", (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        vw.write(f)
    vw.release()
    video_input = "demo_video.mp4"

# 1. Compress with compact 16x latents
!python -m src.cli compress "{video_input}" -o "video.tvid" --gop 48

# 2. Decompress back into an MP4 with audio!
!python -m src.cli decompress "video.tvid" -o "video_restored.mp4"

# 3. Watch the video play in your notebook!
def play_video(path):
    b = open(path, 'rb').read()
    url = "data:video/mp4;base64," + b64encode(b).decode()
    return HTML(f'<video width="600" controls autoplay loop><source src="{url}" type="video/mp4"></video>')

print("\n🎬 Reconstructed Video Playback:")
play_video("video_restored.mp4")

## ⬇️ Download Your Compressed Files

In [ ]:
from google.colab import files
# Uncomment any file you wish to download to your computer:
# files.download("audio.tvid")
# files.download("video.tvid")
# files.download("video_restored.mp4")